# Aula 5 – Organizando e Transformando Dados

## 1. Objetivos da aula
Ao final desta aula, você será capaz de:

* Ordenar dados com `sort_values` e `sort_index`.
* Criar novas colunas usando operações vetorizadas (sem for).
* Usar `assign` para criar colunas de maneira encadeada.
* Arredondar e “podar” valores com `round` e `clip`.
* Reordenar colunas para deixar o DataFrame mais legível.

Essa é a aula em que o DataFrame começa a ficar com cara de relatório profissional.

## 2. Dataset exemplo: Produtos de uma loja
Vamos usar um DataFrame com produtos de uma loja de eletrônicos:

In [ ]:
import pandas as pd

dados_produtos = {
    "produto": [
        "Teclado Mecânico", "Mouse Gamer", "Monitor 24\"", "Headset", "Notebook",
        "Webcam HD", "Cadeira Gamer"
    ],
    "categoria": [
        "Periféricos", "Periféricos", "Monitores", "Periféricos", "Computadores",
        "Periféricos", "Móveis"
    ],
    "custo": [120.0, 60.0, 500.0, 80.0, 2500.0, 70.0, 600.0],
    "preco_venda": [250.0, 150.0, 900.0, 200.0, 4000.0, 160.0, 1300.0],
}
df_prod = pd.DataFrame(dados_produtos)
df_prod

## 3. Ordenando dados com `sort_values` e `sort_index`

### 3.1. Ordenar por uma coluna: `sort_values` 
Queremos ver produtos do mais barato para o mais caro em termos de `preco_venda`:

In [ ]:
df_prod.sort_values("preco_venda")

Por padrão, é crescente. Se quiser em ordem decrescente:

In [ ]:
df_prod.sort_values("preco_venda", ascending=False)

Você também pode ordenar por texto, por exemplo `produto`:

In [ ]:
df_prod.sort_values("produto")

### 3.2. Ordenar por mais de uma coluna
Exemplo: ordenar por categoria e, dentro de cada categoria, por `preco_venda`:

In [ ]:
df_prod.sort_values(["categoria", "preco_venda"])

Ou categoria + preço decrescente:

In [ ]:
df_prod.sort_values(["categoria", "preco_venda"], ascending=[True, False])

### 3.3. Ordenar pelo índice: `sort_index` 
Se você alterou o índice (por exemplo, colocou produto como índice) e quer ordenar por ele:

In [ ]:
df_indexado = df_prod.set_index("produto")
df_indexado.sort_index()

`sort_index` mexe na linha do índice, não nas colunas.

## 4. Criando colunas com operações vetorizadas
Em pandas, você quase nunca usa `for` para calcular uma coluna nova. Você faz operações direto nas colunas.

Queremos criar:
* `lucro = preco_venda - custo` 
* `margem_pct = lucro / preco_venda * 100`

### 4.1. Criando lucro

In [ ]:
df_prod["lucro"] = df_prod["preco_venda"] - df_prod["custo"]
df_prod

### 4.2. Criando margem_pct

In [ ]:
df_prod["margem_pct"] = df_prod["lucro"] / df_prod["preco_venda"] * 100
df_prod

Repare:
* A operação é feita linha a linha, mas você não escreve `for`.
* Isso se chama **operação vetorizada**.

## 5. Usando `assign` para encadear transformações
`assign` é uma forma elegante de criar colunas novas sem quebrar tanto o fluxo da análise.

Exemplo: vamos recriar `lucro` e `margem_pct` usando `assign`:

In [ ]:
df_prod2 = (
    df_prod
      .assign(
          lucro = df_prod["preco_venda"] - df_prod["custo"],
          margem_pct = (df_prod["preco_venda"] - df_prod["custo"]) / df_prod["preco_venda"] * 100
      )
)

df_prod2

Dá para encadear com outras operações, por exemplo:

In [ ]:
df_relatorio = (
    df_prod
      .assign(
          lucro = df_prod["preco_venda"] - df_prod["custo"],
          margem_pct = (df_prod["preco_venda"] - df_prod["custo"]) / df_prod["preco_venda"] * 100
      )
      .sort_values("margem_pct", ascending=False)
)
df_relatorio

Isso já te devolve um DataFrame ordenado do maior para o menor lucro percentual.

## 6. Arredondando valores com `round` 
Números de `margem_pct` podem ficar com muitas casas decimais:

In [ ]:
df_prod["margem_pct"]

### 6.1. round na coluna

In [ ]:
df_prod["margem_pct"] = df_prod["margem_pct"].round(2)

### 6.2. round no DataFrame inteiro

In [ ]:
df_arredondado = df_prod.round(2)

Cuidado: se o DataFrame tiver colunas que não são numéricas, elas ficam como estão.

## 7. Limitando valores com `clip` (útil pra tratamento)
`clip` “corta” valores que estão fora de uma faixa:
* Tudo que for menor que um mínimo vira o mínimo.
* Tudo que for maior que um máximo vira o máximo.

Exemplo prático: Imagine uma coluna de nota de 0 a 10, mas com alguns erros (ex.: 15, -2). Você pode forçar os valores a ficarem no intervalo [0, 10].

In [ ]:
df_notas = pd.DataFrame({
    "aluno": ["Ana", "Bruno", "Carla", "Daniel"],
    "nota": [9.5, 11.0, -1.0, 7.0]
})

df_notas["nota_corrigida"] = df_notas["nota"].clip(lower=0, upper=10)
df_notas

Com isso, você corrige valores inválidos sem precisar de `if` linha a linha.

## 8. Reordenando colunas para deixar a tabela mais legível
Às vezes as colunas ficam numa ordem ruim. Você pode reordenar as colunas manualmente.

### 8.1. Escolher a ordem explicitamente

In [ ]:
colunas = ["produto", "categoria", "custo", "preco_venda", "lucro", "margem_pct"]
df_ordenado = df_prod[colunas]
df_ordenado

Isso não altera o DataFrame original, apenas cria uma “visão” com nova ordem.

## 9. Exemplo completo: relatório de margem de lucro
Vamos juntar tudo:

In [ ]:
df = pd.DataFrame(dados_produtos)

df_relatorio = (
    df
      .assign(
          lucro = df["preco_venda"] - df["custo"],
          margem_pct = ((df["preco_venda"] - df["custo"]) / df["preco_venda"] * 100).round(2)
      )
      .sort_values("margem_pct", ascending=False)
)

colunas = ["produto", "categoria", "custo", "preco_venda", "lucro", "margem_pct"]
df_relatorio = df_relatorio[colunas]

df_relatorio

Você acabou de construir um relatório ordenado por margem de lucro, super comum em cenários de negócios.

## 10. Outro contexto: Ranking de alunos
Agora vamos pra um cenário de notas de alunos:

In [ ]:
dados_alunos = {
    "aluno": ["Ana", "Bruno", "Carla", "Daniel", "Eduarda"],
    "nota_prova1": [8.0, 6.5, 9.0, 5.0, 7.5],
    "nota_prova2": [7.0, 8.0, 8.5, 6.0, 9.0]
}

df_alunos = pd.DataFrame(dados_alunos)
df_alunos

### 10.1. Criando média e status

In [ ]:
df_alunos["media"] = (df_alunos["nota_prova1"] + df_alunos["nota_prova2"]) / 2

Agora, vamos criar um status:
* "Aprovado" se `media >= 7` 
* "Reprovado" caso contrário

Uma forma simples:

In [ ]:
df_alunos["status"] = df_alunos["media"].apply(
    lambda m: "Aprovado" if m >= 7 else "Reprovado"
)

### 10.2. Ordenando pelo melhor desempenho

In [ ]:
df_ranking = df_alunos.sort_values("media", ascending=False)
df_ranking

### 10.3. Arredondando a média

In [ ]:
df_ranking["media"] = df_ranking["media"].round(2)

### 10.4. Reordenando colunas

In [ ]:
colunas = ["aluno", "nota_prova1", "nota_prova2", "media", "status"]
df_ranking = df_ranking[colunas]
df_ranking

Você criou um ranking de alunos, pronto pra ser enviado pra coordenação 😅

## 11. Resumo da aula
Hoje você viu como:

* **Ordenar dados com:** `sort_values` (por colunas, uma ou várias) e `sort_index` (por índice).
* **Criar colunas facilmente** com operações vetorizadas.
* **Usar `assign`** para deixar o código de transformação mais organizado.
* **Arredondar números** com `round`.
* **Limitar valores** em um intervalo com `clip`.
* **Reordenar colunas** para deixar os relatórios mais claros.

Essas técnicas aparecem o tempo todo em análises reais: relatório financeiro, ranking de desempenho, análise de produtos, etc.